# Exercise 1: Tokenization and Subword Vocabulary Analysis
**Referenced Dataset:** Cebuano Bible (`ceb_bible.xlsx`)

In [1]:
# Install dependencies
!uv add tokenizers sentencepiece pandas openpyxl

Resolved 55 packages in 9ms
Checked 50 packages in 2ms


In [ ]:
import pandas as pd
import re
import unicodedata
from collections import Counter

## Task 1: Text Normalization

In [ ]:
# Load the dataset
df = pd.read_excel('ceb_bible.xlsx')

print('Dataset shape (rows, columns):', df.shape)
df.head()

In [ ]:
def is_text(verse):
    """Check if a cell contains valid verse text (not a number, footnote, or empty)."""
    if pd.isna(verse):
        return False
    
    verse_str = str(verse).strip()
    
    # Skip empty strings
    if len(verse_str) == 0:
        return False
    
    # Skip verse numbers or number ranges (e.g., '1', '1-2', '1–3')
    if re.fullmatch(r'[\d\-\u2013]+', verse_str):
        return False
    
    return True

# Step 1: Filter the DataFrame to keep only valid verse rows
valid_mask = df['Verse'].apply(is_text)
valid_verses_df = df[valid_mask]

# Extract verses as a list of strings
verse_list = valid_verses_df['Verse'].astype(str).tolist()

# Combine all verses into a single raw text corpus
corpus_raw = ' '.join(verse_list)

print('Characters (raw):', len(corpus_raw))

In [ ]:
# Step 2: Convert all text to lowercase
text = corpus_raw.lower()

# Step 3: Unicode NFC normalization (ensures consistent character encoding)
text = unicodedata.normalize('NFC', text)

# Step 4: Remove all punctuation marks
text = re.sub(r'[^\w\s]', ' ', text)

# Step 5: Remove all digits / numbers
text = re.sub(r'\d+', '', text)

# Step 6: Collapse multiple spaces or newlines into a single space
text = re.sub(r'\s+', ' ', text)
text = text.strip()

print('Characters (cleaned):', len(text))
print('Sample:', text[:300])

## Task 2: Vocabulary Creation

In [ ]:
# Split cleaned text by whitespace into individual word tokens
tokens = text.split()

# Count the frequency of each unique word
vocab = Counter(tokens)

# Calculate total tokens and vocabulary size
total_tokens = len(tokens)
vocab_size = len(vocab)

print(f'Total word tokens : {total_tokens:,}')
print(f'Vocabulary size   : {vocab_size:,}')

## Task 3: Content Word Analysis

In [ ]:
# Common Cebuano function words (stopwords)
STOPWORDS = {
    'ang', 'sa', 'ug', 'ng', 'si', 'ni', 'kay', 'nga', 'kini', 'siya', 'sila',
    'ko', 'mo', 'niya', 'nato', 'namo', 'ninyo', 'nila', 'kami', 'kamo',
    'ako', 'ikaw', 'kita', 'o', 'aron', 'apan', 'kun', 'kon',
    'mao', 'unya', 'diha', 'diri', 'didto', 'na', 'pa', 'man', 'ba',
    'nang', 'pag', 'mga', 'usa', 'wala', 'adunay', 'walay',
    'ka', 'kang', 'kanako', 'kaniya', 'kanimo', 'kanila', 'kaninyo',
    'busa', 'tungod', 'bisan', 'usab', 'gayod', 'lang',
    'kining', 'niini', 'dinhi', 'karon',
    'dili', 'may', 'ayaw',
    'gikan', 'alang', 'uban', 'ngadto', 'pinaagi', 'hangtod',
    'iyang', 'akong', 'ilang', 'inyong', 'imong', 'atong', 'tanan', 'tanang'
}

# Filter out stopwords and short words (length <= 2)
content_vocab = {}
for word, count in vocab.items():
    if word not in STOPWORDS and len(word) > 2:
        content_vocab[word] = count

# Helper function to extract count for sorting
def get_word_count(item):
    return item[1]

# Sort content words by frequency
sorted_descending = sorted(content_vocab.items(), key=get_word_count, reverse=True)
sorted_ascending = sorted(content_vocab.items(), key=get_word_count, reverse=False)

top10 = sorted_descending[:10]
bottom10 = sorted_ascending[:10]

total_tokens_count = len(tokens)

# Display Top 10 Most Frequent Content Words
print('Top 10 Most Frequent Content Words')
print(f'{"Word":<20} {"Count":>8} {"Rel. Freq (%)":>15}')
print('-' * 45)
for word, count in top10:
    relative_freq = (count / total_tokens_count) * 100
    print(f'{word:<20} {count:>8,} {relative_freq:>14.4f}%')

print()

# Display Bottom 10 Least Frequent Content Words
print('Bottom 10 Least Frequent Content Words')
print(f'{"Word":<20} {"Count":>8} {"Rel. Freq (%)":>15}')
print('-' * 45)
for word, count in bottom10:
    relative_freq = (count / total_tokens_count) * 100
    print(f'{word:<20} {count:>8,} {relative_freq:>14.6f}%')

## Task 4: Research on Subword Tokenization Methods

See the accompanying document file for detailed discussion of:
- **BPE** (Byte Pair Encoding) — Sennrich et al., 2016
- **WordPiece** — Schuster & Nakamura, 2012; used in BERT
- **Unigram Language Model** — Kudo, 2018; used in SentencePiece

## Task 5: Subword Tokenization

In [ ]:
from tokenizers import Tokenizer as HFTokenizer
from tokenizers import pre_tokenizers
from tokenizers.models import BPE, Unigram, WordPiece
from tokenizers.trainers import BpeTrainer, UnigramTrainer, WordPieceTrainer

CORPUS_FILE = '/tmp/ceb_corpus.txt'
VOCAB_SIZE = 1000

# Save cleaned text to file for tokenizer training
with open(CORPUS_FILE, 'w', encoding='utf-8') as f:
    f.write(text)

print(f'Saved corpus to {CORPUS_FILE}')

In [ ]:
# --- 1. Byte Pair Encoding (BPE) ---
# Initialize BPE model
bpe_model = BPE(unk_token='[UNK]')
bpe_tok = HFTokenizer(bpe_model)

# Split by whitespace before subword training
bpe_tok.pre_tokenizer = pre_tokenizers.Whitespace()

# Configure trainer
bpe_trainer = BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=['[UNK]'])

# Train BPE tokenizer
bpe_tok.train([CORPUS_FILE], bpe_trainer)

# Encode corpus to inspect result
bpe_enc = bpe_tok.encode(text)
bpe_total_tokens = len(bpe_enc.tokens)
bpe_vocab_size = bpe_tok.get_vocab_size()

print(f'BPE  | Total tokens: {bpe_total_tokens:,} | Vocab size: {bpe_vocab_size:,}')

In [ ]:
# --- 2. Unigram Language Model ---
# Initialize Unigram model
uni_model = Unigram()
uni_tok = HFTokenizer(uni_model)

# Split by whitespace before subword training
uni_tok.pre_tokenizer = pre_tokenizers.Whitespace()

# Configure trainer
uni_trainer = UnigramTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=['[UNK]'],
    unk_token='[UNK]'
)

# Train Unigram tokenizer
uni_tok.train([CORPUS_FILE], uni_trainer)

# Encode corpus to inspect result
uni_enc = uni_tok.encode(text)
uni_total_tokens = len(uni_enc.tokens)
uni_vocab_size = uni_tok.get_vocab_size()

print(f'UNI  | Total tokens: {uni_total_tokens:,} | Vocab size: {uni_vocab_size:,}')

In [ ]:
# --- 3. WordPiece ---
# Initialize WordPiece model
wp_model = WordPiece(unk_token='[UNK]')
wp_tok = HFTokenizer(wp_model)

# Split by whitespace before subword training
wp_tok.pre_tokenizer = pre_tokenizers.Whitespace()

# Configure trainer
wp_trainer = WordPieceTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=['[UNK]']
)

# Train WordPiece tokenizer
wp_tok.train([CORPUS_FILE], wp_trainer)

# Encode corpus to inspect result
wp_enc = wp_tok.encode(text)
wp_total_tokens = len(wp_enc.tokens)
wp_vocab_size = wp_tok.get_vocab_size()

print(f'WP   | Total tokens: {wp_total_tokens:,} | Vocab size: {wp_vocab_size:,}')

In [ ]:
# Summary table across all tokenizers
summary_data = {
    'Algorithm': ['BPE', 'Unigram', 'WordPiece'],
    'Total Tokens': [
        len(bpe_enc.tokens),
        len(uni_enc.tokens),
        len(wp_enc.tokens)
    ],
    'Vocab Size': [
        bpe_tok.get_vocab_size(),
        uni_tok.get_vocab_size(),
        wp_tok.get_vocab_size()
    ]
}

summary_df = pd.DataFrame(summary_data)
summary_df

## Task 6: Comparative Analysis

In [ ]:
# Three words: rare (freq=1) + morphologically complex
sample_words = [
    'nagapabungolbungol',  # naga-pa- prefix + reduplication of 'bungol' (deaf) = pretending to be deaf
    'gipapanalanginan',    # gi-pa- (causative passive) + panalangin (prayer) + -an (locative) = was caused to be blessed
    'nagabalhinbalhin'     # naga- prefix + reduplication of 'balhin' (move) = kept moving around
]

# Print header
print(f'{"Word":<22} {"BPE":<35} {"Unigram":<35} {"WordPiece"}')
print('-' * 120)

# Display segmentation for each word
for word in sample_words:
    bpe_tokens = bpe_tok.encode(word).tokens
    uni_tokens = uni_tok.encode(word).tokens
    wp_tokens = wp_tok.encode(word).tokens
    
    bpe_segmented = ' | '.join(bpe_tokens)
    uni_segmented = ' | '.join(uni_tokens)
    wp_segmented = ' | '.join(wp_tokens)
    
    print(f'{word:<22} {bpe_segmented:<35} {uni_segmented:<35} {wp_segmented}')